In [1]:
from migration.datasets import create_AIS_dataset

2025-07-03 12:46:36.857264: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-03 12:46:36.857856: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-03 12:46:36.861849: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-03 12:46:36.872371: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1751546796.888349   48577 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1751546796.89

In [2]:
batch_size = 32

In [3]:
import tensorflow as tf
inputs, targets, mmsis, time_starts, time_ends, lengths, mean =  create_AIS_dataset('../../data/ct_2017010203_10_20/ct_2017010203_10_20_train.pkl', 
                   '../../data/ct_2017010203_10_20/mean.pkl',
                   batch_size,
                   99999, # not used lol
                   300,
                   300, 
                   30,
                   72, 
                   shuffle=False,
                   repeat=False)

batch_time_mask = tf.expand_dims(tf.transpose(tf.sequence_mask(lengths, dtype=inputs.dtype)), 2)

Instructions for updating:
Use output_signature instead


2025-07-03 12:46:40.489325: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
2025-07-03 12:46:40.738492: E tensorflow/core/util/util.cc:131] oneDNN supports DT_BOOL only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


In [4]:
mean.shape

TensorShape([702])

In [5]:
print(inputs.shape)
assert (inputs.shape == targets.shape)

(99, 32, 702)


target is just input shifted one place and without mean substracted, both masked positions are cero

In [6]:
batch_time_mask.shape

TensorShape([99, 32, 1])

In [15]:
mmsi_test_data = tf.constant(
    [
        [[1.0, 2.0], 
         [3.0, 4.0],
         [5.0, 6.0],
         [7.0, 8.0]],
         
        [[1.2, 2.2], 
         [3.3, 4.3],
         [0, 0],
         [0, 0]],
    ])

test_length = tf.convert_to_tensor([4,2])
test_mean = tf.convert_to_tensor([1,1.0])

def process_AIS_batch(data, lengths, mean):
    """Create mean-centered and time-major next-step prediction Tensors."""
    data = tf.cast(tf.transpose(data, perm=[1, 0, 2]), dtype=tf.float32)
    mean = tf.cast(mean, dtype=tf.float32)
    lengths = tf.cast(lengths, dtype=tf.int32)
    targets = data

    # Mean center the inputs.
    inputs = data - mean
    # Shift the inputs one step forward in time. Also remove the last
    # timestep so that targets and inputs are the same length.
    inputs = tf.pad(data, [[1, 0], [0, 0], [0, 0]], mode="CONSTANT")[:-1]
    # Mask out unused timesteps.
    inputs *= tf.expand_dims(tf.transpose(
        tf.sequence_mask(lengths, dtype=inputs.dtype)), 2)
    return inputs, targets

test_inputs, test_targets = process_AIS_batch(mmsi_test_data, test_length, test_mean)


In [16]:
test_inputs

<tf.Tensor: shape=(4, 2, 2), dtype=float32, numpy=
array([[[0. , 0. ],
        [0. , 0. ]],

       [[1. , 2. ],
        [1.2, 2.2]],

       [[3. , 4. ],
        [0. , 0. ]],

       [[5. , 6. ],
        [0. , 0. ]]], dtype=float32)>

In [17]:
test_targets

<tf.Tensor: shape=(4, 2, 2), dtype=float32, numpy=
array([[[1. , 2. ],
        [1.2, 2.2]],

       [[3. , 4. ],
        [3.3, 4.3]],

       [[5. , 6. ],
        [0. , 0. ]],

       [[7. , 8. ],
        [0. , 0. ]]], dtype=float32)>

In [7]:
print(mmsis)
print(len(mmsis))

tf.Tensor(
[304655000 304655000 236631000 236631000 236187000 236187000 311044200
 311044200 311044200 305555000 305555000 305808000 246061000 244366000
 244366000 244366000 244796000 244796000 244796000 538090070 538090070
 304407000 304407000 304407000 304407000 304407000 304407000 564919000
 636017516 636017516 250001109 250001109], shape=(32,), dtype=int32)
32


In [8]:
print(time_starts)
print(len(time_starts))

tf.Tensor(
[1.4832288e+09 1.4891593e+09 1.4834349e+09 1.4875086e+09 1.4832288e+09
 1.4837780e+09 1.4840634e+09 1.4868408e+09 1.4876600e+09 1.4832288e+09
 1.4879027e+09 1.4832288e+09 1.4861757e+09 1.4832288e+09 1.4856737e+09
 1.4889797e+09 1.4832289e+09 1.4851105e+09 1.4891576e+09 1.4832289e+09
 1.4849838e+09 1.4832289e+09 1.4838957e+09 1.4845748e+09 1.4861736e+09
 1.4868836e+09 1.4885906e+09 1.4832289e+09 1.4832289e+09 1.4846417e+09
 1.4832291e+09 1.4837313e+09], shape=(32,), dtype=float32)
32


In [20]:
print(time_ends)
assert len(time_ends) == batch_size


tf.Tensor(
[1.4832692e+09 1.4891867e+09 1.4834588e+09 1.4875341e+09 1.4832477e+09
 1.4837946e+09 1.4841032e+09 1.4868648e+09 1.4877042e+09 1.4832522e+09
 1.4879459e+09 1.4832543e+09 1.4862144e+09 1.4832520e+09 1.4857152e+09
 1.4890097e+09 1.4832442e+09 1.4851465e+09 1.4891864e+09 1.4832499e+09
 1.4850057e+09 1.4832486e+09 1.4839500e+09 1.4846336e+09 1.4862167e+09
 1.4869039e+09 1.4886222e+09 1.4832502e+09 1.4832557e+09 1.4846588e+09
 1.4832588e+09 1.4837731e+09], shape=(32,), dtype=float32)


In [30]:
time_ends

<tf.Tensor: shape=(32,), dtype=float32, numpy=
array([1.4832692e+09, 1.4891867e+09, 1.4834588e+09, 1.4875341e+09,
       1.4832477e+09, 1.4837946e+09, 1.4841032e+09, 1.4868648e+09,
       1.4877042e+09, 1.4832522e+09, 1.4879459e+09, 1.4832543e+09,
       1.4862144e+09, 1.4832520e+09, 1.4857152e+09, 1.4890097e+09,
       1.4832442e+09, 1.4851465e+09, 1.4891864e+09, 1.4832499e+09,
       1.4850057e+09, 1.4832486e+09, 1.4839500e+09, 1.4846336e+09,
       1.4862167e+09, 1.4869039e+09, 1.4886222e+09, 1.4832502e+09,
       1.4832557e+09, 1.4846588e+09, 1.4832588e+09, 1.4837731e+09],
      dtype=float32)>

In [31]:
assert ((time_ends-time_starts).numpy() > 0).all()

In [33]:
print(lengths)
assert (lengths <= inputs.shape[0]).numpy().all()
assert len(lengths) == batch_size

tf.Tensor(
[68 46 41 43 32 28 67 41 74 40 73 43 65 39 70 51 26 61 49 36 37 34 91 99
 73 35 53 36 45 29 50 70], shape=(32,), dtype=int32)
